In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras import layers

2026-02-04 16:22:34.023556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770243754.041932  129246 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770243754.047419  129246 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770243754.061279  129246 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770243754.061305  129246 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770243754.061307  129246 computation_placer.cc:177] computation placer alr

In [11]:
data1 = pd.DataFrame({
    "edad": np.random.randint(21, 65, 1000),
    "sexo": np.random.choice(["M", "F"], 1000),
    "ingreso": np.random.lognormal(mean=10, sigma=0.5, size=1000),
    "profesion": np.random.choice(
        ["ingeniero", "doctor", "abogado", "profesor"], 1000
    )
})

data2 = pd.DataFrame({
    "edad": np.random.randint(18, 65, 100),
    "sexo": np.random.choice(["M", "F"], 100),
    "ingreso": 0,
    "profesion": np.random.choice(
        ["desempleado"], 100
    )
})

data3 = pd.DataFrame({
    "edad": np.random.randint(3, 21, 200),
    "sexo": np.random.choice(["M", "F"], 200),
    "ingreso": 0,
    "profesion": np.random.choice(
        ["estudiante"], 200
    )
})

data = pd.concat([data1, data2, data3])
data.reset_index(drop=True, inplace=True)
data

,edad,sexo,ingreso,profesion
0,53,F,23037.246657,ingeniero
1,25,F,16955.906231,doctor
2,56,M,71974.624508,abogado
3,42,F,27040.900377,doctor
4,24,F,19223.496028,ingeniero
...,...,...,...,...
1295,15,F,0.000000,estudiante
1296,13,F,0.000000,estudiante
1297,3,F,0.000000,estudiante
1298,20,M,0.000000,estudiante


In [12]:
categorical_cols = ["sexo", "profesion"]
continuous_cols = ["edad", "ingreso"]

# Log transform ingreso
data["ingreso"] = np.log(data["ingreso"] + 1)

ohe = OneHotEncoder(sparse_output=False)
scaler = StandardScaler()

X_cat = ohe.fit_transform(data[categorical_cols])
X_cont = scaler.fit_transform(data[continuous_cols])

X = np.concatenate([X_cont, X_cat], axis=1)
X = X.astype("float32")

data_dim = X.shape[1]

In [13]:
cat_dims = [len(ohe.categories_[i]) for i in range(len(categorical_cols))]
cond_dim = sum(cat_dims)

def sample_condition(batch_size):
    cond = np.zeros((batch_size, cond_dim))
    for i, dim in enumerate(cat_dims):
        idx = np.random.randint(0, dim, batch_size)
        start = sum(cat_dims[:i])
        cond[np.arange(batch_size), start + idx] = 1
    return tf.convert_to_tensor(cond, dtype=tf.float32)


In [19]:
print(cat_dims, cond_dim)

[2, 6] 8


In [21]:
sample_condition(5)

<tf.Tensor: shape=(5, 8), dtype=float32, numpy=
array([[1., 0., 0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0., 1., 0.],
       [1., 0., 1., 0., 0., 0., 0., 0.]], dtype=float32)>

In [14]:
latent_dim = 128

def build_generator():
    z = layers.Input(shape=(latent_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([z, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(data_dim)(x)

    return tf.keras.Model([z, cond], out)


In [15]:
def build_discriminator():
    x_in = layers.Input(shape=(data_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([x_in, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(1)(x)

    return tf.keras.Model([x_in, cond], out)


In [16]:
def gradient_penalty(discriminator, real, fake, cond):
    alpha = tf.random.uniform([real.shape[0], 1], 0., 1.)
    interpolated = alpha * real + (1 - alpha) * fake

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = discriminator([interpolated, cond], training=True)

    grads = tape.gradient(pred, interpolated)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1))
    return tf.reduce_mean((norm - 1.0) ** 2)


In [17]:
generator = build_generator()
discriminator = build_discriminator()

g_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)

batch_size = 256
epochs = 300
lambda_gp = 10
critic_steps = 5

dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(2000).batch(batch_size)

for epoch in range(epochs):
    for real_batch in dataset:
        for _ in range(critic_steps):
            z = tf.random.normal([real_batch.shape[0], latent_dim])
            cond = sample_condition(real_batch.shape[0])

            with tf.GradientTape() as tape:
                fake = generator([z, cond], training=True)
                d_real = discriminator([real_batch, cond], training=True)
                d_fake = discriminator([fake, cond], training=True)

                gp = gradient_penalty(discriminator, real_batch, fake, cond)
                d_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + lambda_gp * gp

            grads = tape.gradient(d_loss, discriminator.trainable_variables)
            d_opt.apply_gradients(zip(grads, discriminator.trainable_variables))

        z = tf.random.normal([real_batch.shape[0], latent_dim])
        cond = sample_condition(real_batch.shape[0])

        with tf.GradientTape() as tape:
            fake = generator([z, cond], training=True)
            g_loss = -tf.reduce_mean(discriminator([fake, cond], training=True))

        grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(grads, generator.trainable_variables))

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | D: {d_loss.numpy():.3f} | G: {g_loss.numpy():.3f}")


Epoch 0 | D: 2.858 | G: -0.397
Epoch 50 | D: -0.561 | G: -0.140
Epoch 100 | D: -0.949 | G: -0.455
Epoch 150 | D: -0.639 | G: 0.132
Epoch 200 | D: -0.780 | G: 0.796


2026-02-04 16:39:39.296293: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 250 | D: -0.881 | G: 2.040


In [18]:
z = tf.random.normal([1000, latent_dim])
cond = sample_condition(1000)

synthetic = generator([z, cond], training=False).numpy()

X_cont_syn = scaler.inverse_transform(synthetic[:, :len(continuous_cols)])
X_cat_syn = ohe.inverse_transform(synthetic[:, len(continuous_cols):])

synthetic_df = pd.DataFrame(
    np.column_stack([X_cont_syn, X_cat_syn]),
    columns=continuous_cols + categorical_cols
)

synthetic_df['ingreso'] = np.exp(synthetic_df['ingreso'].astype(float)) 

print(synthetic_df.head(50))


         edad        ingreso sexo   profesion
0   47.502201   33314.968937    F     abogado
1   44.299423    5608.582962    F   ingeniero
2   59.804688   63573.122651    F      doctor
3   63.307991   30223.117207    F     abogado
4   23.980551      34.648402    M   ingeniero
5   51.189236   46844.911252    M      doctor
6   34.935223   85654.745700    M   ingeniero
7    4.166857       0.002921    F  estudiante
8   59.123306    6616.818584    M   ingeniero
9   42.317055    4681.478630    F      doctor
10    53.2481  101382.538464    F    profesor
11   6.252794      10.601303    F  estudiante
12  39.080002  130521.974038    M   ingeniero
13  28.213209    8460.546271    M      doctor
14  30.550528   20706.182356    F    profesor
15  37.833019     650.593645    M   ingeniero
16   37.29496   56226.760846    M   ingeniero
17  27.224754    4873.550969    F   ingeniero
18  35.674358   54180.631627    M      doctor
19  36.415539    3013.270426    M    profesor
20  41.247486   24802.223450    M 